# OPUS Baseline Benchmark (Random Selection)

Runs the **baseline** (random data selection) training benchmark.

**What this measures:** End-to-end step timing with random candidate selection (no OPUS scoring). Provides `T_base` for computing OPUS overhead.

**After this notebook**, run the OPUS benchmark notebook, then compare.

## Step 1: Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No CUDA GPU detected. DeepSpeed requires NVIDIA GPU.")

## Step 2: Upload Codebase to Colab Runtime

**First, zip the folder on your Mac** (only needed once):
```bash
cd ~/Downloads && zip -r OpusImplementation_updated.zip OpusImplementation_updated/
```

Then run the cell below. It will:
1. Show a file picker — select your zip
2. Wait for the upload to finish
3. Extract automatically

**If the widget doesn't work**, use the fallback cell after it instead.

In [ ]:
import os, sys, io, zipfile, time, shutil

PROJECT_DIR = "/content/OpusImplementation_updated"
MARKER = os.path.join(PROJECT_DIR, "requirements.txt")

# Clean up any stale/partial directory from previous failed attempts
if os.path.exists(PROJECT_DIR) and not os.path.isfile(MARKER):
    print(f"Removing stale directory: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

if os.path.isfile(MARKER):
    print(f"Project already exists at {PROJECT_DIR}, skipping upload.")
else:
    import ipywidgets as widgets
    from IPython.display import display

    print("Select OpusImplementation_updated.zip using the button below.")
    uploader = widgets.FileUpload(accept=".zip", multiple=False)
    display(uploader)

    # Wait for user to pick a file (poll every 2s, timeout 5 min)
    for i in range(150):
        time.sleep(2)
        # ipywidgets v8: value is a tuple; v7: value is a dict
        if isinstance(uploader.value, (list, tuple)) and len(uploader.value) > 0:
            break
        elif isinstance(uploader.value, dict) and len(uploader.value) > 0:
            break
    else:
        raise TimeoutError("No file selected after 5 minutes. Re-run this cell.")

    # Extract content (handle v7 and v8 API differences)
    if isinstance(uploader.value, dict):
        # v7: {filename: {content: bytes}}
        fname = list(uploader.value.keys())[0]
        raw = uploader.value[fname]["content"]
    else:
        # v8: tuple of dicts with 'name', 'content'
        raw = bytes(uploader.value[0]["content"])
        fname = uploader.value[0]["name"]

    print(f"Received: {fname} ({len(raw) / 1e6:.1f} MB)")
    print("Extracting...")
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        z.extractall("/content/")

    if not os.path.isfile(MARKER):
        # Maybe the zip has the folder nested differently — list top-level
        print("WARNING: requirements.txt not at expected path. Zip contents:")
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            for name in z.namelist()[:20]:
                print(f"  {name}")
        raise FileNotFoundError(
            f"Expected {MARKER}. "
            "Make sure the zip contains 'OpusImplementation_updated/' as the top-level folder."
        )

    print("Extraction complete!")

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

OUTPUT_DIR = os.path.join(PROJECT_DIR, "benchmark_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DATA_DIR = os.path.join(PROJECT_DIR, "synth_local_en")

print(f"\nProject dir: {PROJECT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Files:")
!ls {PROJECT_DIR}

In [ ]:
# ============================================================
# FALLBACK: If the widget above doesn't work, use this instead.
# On your Mac terminal, run:
#   cd ~/Downloads && python3 -m http.server 8888
# Then use ngrok or paste a direct download URL below.
# ============================================================
# Uncomment ONE of these:

# Option A: gdown from Google Drive (upload zip to Drive, share link, paste here)
# !pip install -q gdown
# !gdown --fuzzy "YOUR_DRIVE_SHARE_LINK_HERE" -O /content/opus.zip
# !unzip -qo /content/opus.zip -d /content/ && rm /content/opus.zip

# Option B: wget from any direct download URL
# !wget -q "YOUR_DIRECT_URL_HERE" -O /content/opus.zip
# !unzip -qo /content/opus.zip -d /content/ && rm /content/opus.zip

## Step 3: Install Dependencies

In [ ]:
!pip install transformers==4.40.0 datasets deepspeed ninja triton huggingface_hub flash-linear-attention 2>&1 | tail -5
print("\nDependencies installed.")

## Step 3b: Verify Import Chain

Checks that all modules the trainer needs can be imported from the Colab runtime.

In [ ]:
import os, sys

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

errors = []

# 1. Core files must exist
for f in ["recurrence_model_1b.py", "recurrence_model_70b.py", "training.py",
           "data.py", "liger_ops.py", "kernels/__init__.py",
           "production/trainer.py", "production/train_prod.py",
           "production/config.py", "production/opus/__init__.py",
           "production/deepspeed_zero2_config.json"]:
    path = os.path.join(PROJECT_DIR, f)
    status = "OK" if os.path.isfile(path) else "MISSING"
    if status == "MISSING":
        errors.append(f)
    print(f"  [{status}] {f}")

# 2. Test key Python imports
print("\nImport checks:")
for mod_name in ["production.config", "production.opus", "data", "training",
                  "recurrence_model_1b", "liger_ops", "kernels"]:
    try:
        __import__(mod_name)
        print(f"  [OK] import {mod_name}")
    except Exception as e:
        errors.append(f"import {mod_name}: {e}")
        print(f"  [FAIL] import {mod_name}: {e}")

# 3. Test DeepSpeed
print("\nDeepSpeed check:")
try:
    import deepspeed
    print(f"  [OK] deepspeed {deepspeed.__version__}")
except Exception as e:
    errors.append(f"deepspeed: {e}")
    print(f"  [FAIL] {e}")

if errors:
    print(f"\n{len(errors)} issue(s) found — fix before running benchmark.")
else:
    print("\nAll checks passed. Ready to run.")

## Step 4: Download Training Data (PleIAs/SYNTH shard)

In [ ]:
import os

if not os.path.exists(os.path.join(DATA_DIR, "dataset_info.json")):
    !python {PROJECT_DIR}/scripts/download_synth_shard.py -o {DATA_DIR}
    print("Synth data downloaded.")
else:
    print(f"Synth data already exists at {DATA_DIR}, skipping.")

## Step 5: Create Matched Baseline Config

Identical to the OPUS notebook **except** `selection_mode = "random"`.

Both notebooks use: 1B model, vocab 131072, seq_len 512, candidate_multiplier 32, 100 steps.

In [ ]:
import json, os

config = {
    "model": {
        "target": "1b",
        "embedding_type": "standard",
        "vocab_size": 131072,
        "hidden_size": None,
        "num_layers": None,
        "num_real_experts": None,
        "num_null_experts": None,
        "top_k": None,
        "data_sparsity": None
    },
    "data": {
        "dataset_name": "PleIAs/SYNTH",
        "local_path": DATA_DIR,
        "filter_language": "en",
        "seed": 42,
        "num_workers": 2
    },
    "train": {
        "total_steps": 100,
        "learning_rate": 0.0002,
        "weight_decay": 0.0,
        "betas": [0.9, 0.95],
        "eps": 1e-08,
        "grad_clip": 1.0,
        "seq_len_train": 512,
        "micro_batch_size": 1,
        "grad_accum_steps": 1,
        "checkpoint_every": 250,
        "log_every": 10,
        "checkpoint_dir": os.path.join(OUTPUT_DIR, "checkpoints_baseline"),
        "step_metrics_log_path": os.path.join(OUTPUT_DIR, "step_timing_baseline.jsonl"),
        "step_metrics_log_every": 1,
        "deepspeed_config": "production/deepspeed_zero2_config.json",
        "use_bf16": True
    },
    "opus": {
        "enabled": True,
        "selection_mode": "random",
        "candidate_multiplier": 32,
        "selection_ratio": 0.5,
        "score_seq_len": 512,
        "proxy_batch_size": 8,
        "sketch_dim": 8192,
        "temperature": 0.9,
        "sketch_seed": 42,
        "fallback_random_on_error": True,
        "max_selector_time_s": 30.0,
        "include_embeddings": False,
        "include_lm_head": False,
        "score_dtype": "bf16",
        "track_nonfinite_stats": False,
        "zero2_exact_global_scoring": True,
        "strict_shard_preconditioner": True,
        "benchmark_dual_mode": False,
        "benchmark_every": 100,
        "benchmark_warmup_steps": 20,
        "benchmark_log_path": os.path.join(OUTPUT_DIR, "selector_benchmark_baseline.jsonl")
    }
}

config_path = os.path.join(PROJECT_DIR, "config_colab_baseline.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config written to: {config_path}")
print(f"Selection mode:    {config['opus']['selection_mode']}")
print(f"Total steps:       {config['train']['total_steps']}")
print(f"Seq len:           {config['train']['seq_len_train']}")
print(f"Vocab size:        {config['model']['vocab_size']}")

## Step 6: Run Baseline Benchmark

Runs production trainer with **random selection** (no OPUS scoring overhead).

100 steps at ~5-15s each = ~8-25 minutes.

In [ ]:
import subprocess, os

log_file = os.path.join(OUTPUT_DIR, "baseline_run.log")

cmd = (
    f"cd '{PROJECT_DIR}' && "
    f"PYTHONPATH='{PROJECT_DIR}' "
    f"deepspeed --num_gpus 1 "
    f"production/train_prod.py "
    f"--config config_colab_baseline.json"
)

print(f"Running: {cmd}")
print(f"Logging to: {log_file}")
print("=" * 60)

process = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

with open(log_file, "w") as f:
    for line in process.stdout:
        print(line, end="")
        f.write(line)

process.wait()
print(f"\nProcess exited with code: {process.returncode}")

## Step 7: View Results

In [ ]:
import json, statistics, os

log_path = os.path.join(OUTPUT_DIR, "step_timing_baseline.jsonl")
warmup = 20

steps = []
with open(log_path, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            steps.append(json.loads(line))

print(f"Total steps logged: {len(steps)}")
print(f"\n--- First 5 steps ---")
for s in steps[:5]:
    print(json.dumps(s, indent=2))

post_warmup = [s for s in steps if s["step"] >= warmup]
if post_warmup:
    times = [s["step_total_s"] for s in post_warmup]
    print(f"\n--- Baseline Timing (post warmup, {len(times)} steps) ---")
    print(f"  Mean:   {statistics.fmean(times):.3f} s/step")
    print(f"  Median: {statistics.median(times):.3f} s/step")
    sorted_times = sorted(times)
    p90_idx = min(len(sorted_times)-1, int(0.9 * (len(sorted_times)-1)))
    print(f"  P90:    {sorted_times[p90_idx]:.3f} s/step")
    print(f"  Min:    {min(times):.3f} s/step")
    print(f"  Max:    {max(times):.3f} s/step")
else:
    print("No steps after warmup!")

print(f"\nTiming log saved at: {log_path}")
print("This log is automatically picked up by the OPUS notebook for overhead comparison.")